# MLP Classifier & Threshold Detector Baseline (Task A2: R2.3, R2.4)

Two focused additions to the main comparison, **appended** to the existing result CSVs
rather than re-running the other 9 models (chosen as the less invasive option per the
task spec):

- **R2.3** — Reviewer 2 asked whether a neural-network model was tried. `"MLP"`
  (`MLPClassifier` in a scaling pipeline) is added to `eval_protocol.MODEL_CONFIGS` and
  run through the same primary protocol as the other 9 models, for all 3 tasks and both
  feature sets, plus the V1a/V1b locked hold-out designs (Task A1).
- **R2.4** — Reviewer 1 called the 9-classifier comparison "insufficient as a
  system-level baseline" for detection. A depth-1 decision tree on `mean_rssi` (a single
  learned RSSI threshold) is added as a formal, minimal baseline for Attack Detection,
  with the same false-alarm/detection-sensitivity breakdown used for KNN in Task A1's
  V5/F3.

Requires `results/table1_primary.csv`, `results/best_params.csv` and
`results/final_holdout.csv` from `02_final_comparison.ipynb` / `04_validation_robustness.ipynb`.

## 1. Imports & Configuration

In [1]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import eval_protocol as ep

REPO_ROOT = ep._find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

table1_df = pd.read_csv(RESULTS_DIR / "table1_primary.csv", index_col="Model")
final_holdout_df = pd.read_csv(RESULTS_DIR / "final_holdout.csv")
print(f"Loaded table1_primary.csv ({table1_df.shape[0]} models) and final_holdout.csv ({len(final_holdout_df)} rows)")
print("MLP already in MODEL_CONFIGS:", "MLP" in ep.MODEL_CONFIGS)

Loaded table1_primary.csv (9 models) and final_holdout.csv (45 rows)
MLP already in MODEL_CONFIGS: True


## 2. Load Dataset

In [2]:
raw = ep.load_raw()
dataset, info = ep.build_dataset(raw, freq="5s")
dataset = ep.add_fe(dataset)
assert info.n_rows == 2131 and info.n_attack_rows == 571 and info.n_sessions_attack == 48
print(f"Dataset: {info.n_rows} rows, {info.n_attack_rows} attack rows, {info.n_sessions_attack} attack sessions")

Dataset: 2131 rows, 571 attack rows, 48 attack sessions


## 3. R2.3 — MLP in the Main Comparison

Nested CV for `"MLP"` only, same primary protocol as `02_final_comparison.ipynb`, for
all 3 tasks x 2 feature sets (6 runs) — **the other 9 models are not re-run.** MLP's
row is appended to `table1_primary.csv` (and `per_fold_scores.csv` / `best_params.csv`
for consistency with the per-fold data the other 9 models already have there).

In [3]:
FEATURE_SETS = {"Base (4)": ep.FEATURES_BASE, "FE (13)": ep.FEATURES_FE}

mlp_results_rows = []
mlp_per_fold_rows = []
mlp_best_params_rows = []
mlp_oof = {}

t_start = time.time()
for fs_label, feat_cols in FEATURE_SETS.items():
    for task in ep.TASKS_ORDER:
        spec = ep.TASKS[task]
        sub_t = dataset[spec["rows"](dataset)].reset_index(drop=True)
        X = sub_t[feat_cols]
        y = sub_t[spec["y_col"]]
        outer_cv, groups = ep.get_outer_cv_and_groups("primary", task, sub_t)
        is_multiclass = task in ("point", "antenna")

        t0 = time.time()
        res = ep.nested_cv("MLP", X, y, groups, outer_cv, ep.group_kfold_inner_factory(5), is_multiclass=is_multiclass)
        summ = ep.summarize(res, task)
        mlp_oof[(fs_label, task)] = res

        mlp_results_rows.append({
            "Model": "MLP", "Feature Set": fs_label, "Task": spec["label"],
            "Acc Mean": summ["acc_mean"], "Acc Std": summ["acc_std"],
            "Pooled Acc": summ["pooled_acc"], "Macro F1": summ["macro_f1"],
        })
        for fold_i, score in enumerate(res["fold_scores"]):
            mlp_per_fold_rows.append({"Model": "MLP", "Feature Set": fs_label, "Task": spec["label"], "Fold": fold_i, "Accuracy": score})
        for fold_i, params in enumerate(res["best_params_per_fold"]):
            mlp_best_params_rows.append({"Model": "MLP", "Feature Set": fs_label, "Task": spec["label"], "Fold": fold_i, "Best Params": params})

        print(f"[{fs_label:>8}] {spec['label']:<20} MLP pooled_acc={summ['pooled_acc']:.3f} ({time.time()-t0:.1f}s)")

print(f"\nMLP main-table runs: {len(mlp_results_rows)}  |  Total time: {time.time()-t_start:.1f}s")
mlp_results_df = pd.DataFrame(mlp_results_rows)

[Base (4)] Attack Detection     MLP pooled_acc=0.988 (5.7s)


[Base (4)] Point Prediction     MLP pooled_acc=0.688 (3.6s)


[Base (4)] Antenna Prediction   MLP pooled_acc=0.280 (4.4s)


[ FE (13)] Attack Detection     MLP pooled_acc=0.987 (4.0s)


[ FE (13)] Point Prediction     MLP pooled_acc=0.718 (4.0s)


[ FE (13)] Antenna Prediction   MLP pooled_acc=0.282 (5.6s)

MLP main-table runs: 6  |  Total time: 27.3s


In [4]:
def build_mlp_table1_row(results_df):
    row = {"Model": "MLP"}
    for task in ep.TASKS_ORDER:
        task_label = ep.TASKS[task]["label"]
        base = results_df[(results_df.Task == task_label) & (results_df["Feature Set"] == "Base (4)")].iloc[0]
        fe = results_df[(results_df.Task == task_label) & (results_df["Feature Set"] == "FE (13)")].iloc[0]
        row[f"{task_label} Pooled Acc (Base)"] = base["Pooled Acc"]
        row[f"{task_label} Pooled Acc (FE)"] = fe["Pooled Acc"]
        row[f"{task_label} Acc Mean+-Std (Base)"] = f"{base['Acc Mean']:.3f}+/-{base['Acc Std']:.3f}"
        row[f"{task_label} Acc Mean+-Std (FE)"] = f"{fe['Acc Mean']:.3f}+/-{fe['Acc Std']:.3f}"
        row[f"{task_label} Macro F1 (Base)"] = round(base["Macro F1"], 4)
        row[f"{task_label} Macro F1 (FE)"] = round(fe["Macro F1"], 4)
        row[f"{task_label} Delta Pooled Acc (FE-Base)"] = round(fe["Pooled Acc"] - base["Pooled Acc"], 4)
    return pd.Series(row).drop("Model")

table1_df = table1_df.drop(index="MLP", errors="ignore")  # idempotent re-run
table1_df.loc["MLP"] = build_mlp_table1_row(mlp_results_df)
table1_df.to_csv(RESULTS_DIR / "table1_primary.csv")

per_fold_df = pd.read_csv(RESULTS_DIR / "per_fold_scores.csv")
per_fold_df = per_fold_df[per_fold_df.Model != "MLP"]
per_fold_df = pd.concat([per_fold_df, pd.DataFrame(mlp_per_fold_rows)], ignore_index=True)
per_fold_df.to_csv(RESULTS_DIR / "per_fold_scores.csv", index=False)

best_params_df = pd.read_csv(RESULTS_DIR / "best_params.csv")
best_params_df = best_params_df[best_params_df.Model != "MLP"]
best_params_df = pd.concat([best_params_df, pd.DataFrame(mlp_best_params_rows)], ignore_index=True)
best_params_df.to_csv(RESULTS_DIR / "best_params.csv", index=False)

print("MLP row in table1_primary.csv:")
table1_df.loc[["MLP"]].round(4)

MLP row in table1_primary.csv:


,Attack Detection Pooled Acc (Base),Attack Detection Pooled Acc (FE),Attack Detection Acc Mean+-Std (Base),Attack Detection Acc Mean+-Std (FE),Attack Detection Macro F1 (Base),Attack Detection Macro F1 (FE),Attack Detection Delta Pooled Acc (FE-Base),Point Prediction Pooled Acc (Base),Point Prediction Pooled Acc (FE),Point Prediction Acc Mean+-Std (Base),...,Point Prediction Macro F1 (Base),Point Prediction Macro F1 (FE),Point Prediction Delta Pooled Acc (FE-Base),Antenna Prediction Pooled Acc (Base),Antenna Prediction Pooled Acc (FE),Antenna Prediction Acc Mean+-Std (Base),Antenna Prediction Acc Mean+-Std (FE),Antenna Prediction Macro F1 (Base),Antenna Prediction Macro F1 (FE),Antenna Prediction Delta Pooled Acc (FE-Base)
Model,,,,,,,,,,,,,,,,,,,,,
MLP,0.9883,0.9869,0.988+/-0.003,0.987+/-0.005,0.9851,0.9833,-0.0014,0.6883,0.718,0.689+/-0.142,...,0.6892,0.7221,0.0298,0.2802,0.282,0.276+/-0.145,0.282+/-0.089,0.2415,0.2598,0.0018


### MLP in the V1a/V1b locked hold-out (Task A1, F1 convention)

For each task/design, MLP's dev-only nested-CV pooled accuracy (`select_best_model_on_dev`'s
scoring, computed fresh for MLP alone — the other 9 models' dev scores are already
stored in `final_holdout.csv` and are not recomputed) is compared against the existing
`dev_pooled_acc` values for that task/design. If MLP's dev score is higher, it becomes
the new `selected_on_dev` model (the previous one is un-flagged); otherwise it is added
as a reported-but-not-selected row, consistent with F1: the comparison never looks at
the held-out test rows.

In [5]:
FS_V1 = "FE (13)"
feat_cols_v1 = ep.FEATURES_FE

dev_mask_a = dataset["block"] <= 5
test_mask_a = dataset["block"] == 6
v1b_sessions = ep.select_balanced_holdout_sessions(raw)
test_session_ids = set(v1b_sessions["session"]) | set(v1b_sessions["gap_session"])
test_mask_b = dataset["session"].isin(test_session_ids)
dev_mask_b = ~test_mask_b

V1_DESIGNS = {
    "V1a temporal (block 6 held out)": {"dev_mask": dev_mask_a, "test_mask": test_mask_a, "tasks": ["attack", "point"]},
    "V1b balanced session hold-out": {"dev_mask": dev_mask_b, "test_mask": test_mask_b, "tasks": ["attack", "point", "antenna"]},
}

new_holdout_rows = []
flip_index = []  # (task, design) pairs where MLP becomes the new selected_on_dev

for design_label, design in V1_DESIGNS.items():
    for task in design["tasks"]:
        spec = ep.TASKS[task]
        task_label = spec["label"]
        is_multiclass = task != "attack"
        group_col = ep.PROTOCOLS["primary"].group_col_by_task[task]

        row_mask = spec["rows"](dataset)
        dev_mask = row_mask & design["dev_mask"]
        test_mask = row_mask & design["test_mask"]

        X_dev = dataset.loc[dev_mask, feat_cols_v1]
        y_dev = dataset.loc[dev_mask, spec["y_col"]]
        groups_dev = dataset.loc[dev_mask, group_col].to_numpy()
        sessions_dev = dataset.loc[dev_mask, "session"].to_numpy()

        X_test = dataset.loc[test_mask, feat_cols_v1]
        y_test = dataset.loc[test_mask, spec["y_col"]]
        sessions_test = dataset.loc[test_mask, "session"].to_numpy()

        t0 = time.time()
        mlp_outer_cv = ep.LeaveOneGroupOut()
        mlp_dev_res = ep.nested_cv("MLP", X_dev, y_dev, groups_dev, mlp_outer_cv, ep.group_kfold_inner_factory(5), is_multiclass=is_multiclass)
        mlp_dev_score = ep.summarize(mlp_dev_res, task)["pooled_acc"]

        existing = final_holdout_df[(final_holdout_df.task == task_label) & (final_holdout_df.design == design_label)]
        current_best_dev = existing["dev_pooled_acc"].max()
        becomes_selected = mlp_dev_score > current_best_dev

        holdout_res = ep.locked_holdout_eval(
            "MLP", X_dev, y_dev, groups_dev, sessions_dev,
            X_test, y_test, sessions_test, is_multiclass=is_multiclass, task=task,
        )
        ci = ep.cluster_bootstrap_ci(y_test.to_numpy(), holdout_res["y_pred"], sessions_test, n_boot=2000, seed=0)

        new_holdout_rows.append({
            "task": task_label, "design": design_label, "model": "MLP",
            "selected_on_dev": bool(becomes_selected), "dev_pooled_acc": mlp_dev_score,
            "accuracy": holdout_res["pooled_acc"], "macro_f1": holdout_res["macro_f1"],
            "precision": holdout_res.get("precision"), "recall": holdout_res.get("recall"), "fpr": holdout_res.get("fpr"),
            "n_test_rows": holdout_res["n_test_rows"], "n_test_sessions": holdout_res["n_test_sessions"],
            "ci_low": ci["ci_low"], "ci_high": ci["ci_high"],
        })
        if becomes_selected:
            flip_index.append((task_label, design_label))
        print(f"[V1 MLP] {design_label[:12]:<12} {task_label:<20} dev_acc={mlp_dev_score:.3f} "
              f"(current best dev={current_best_dev:.3f}) selected={becomes_selected} test_acc={holdout_res['pooled_acc']:.3f} ({time.time()-t0:.1f}s)")

final_holdout_df = final_holdout_df[final_holdout_df.model != "MLP"]
for task_label, design_label in flip_index:
    mask = (final_holdout_df.task == task_label) & (final_holdout_df.design == design_label)
    final_holdout_df.loc[mask, "selected_on_dev"] = False
final_holdout_df = pd.concat([final_holdout_df, pd.DataFrame(new_holdout_rows)], ignore_index=True)
final_holdout_df.to_csv(RESULTS_DIR / "final_holdout.csv", index=False)

print(f"\nMLP became selected_on_dev for: {flip_index if flip_index else 'no task/design (existing champions still win)'}")
final_holdout_df[final_holdout_df.model == "MLP"][["task", "design", "selected_on_dev", "dev_pooled_acc", "accuracy", "ci_low", "ci_high"]].round(4)

[V1 MLP] V1a temporal Attack Detection     dev_acc=0.984 (current best dev=0.989) selected=False test_acc=0.997 (3.1s)


[V1 MLP] V1a temporal Point Prediction     dev_acc=0.680 (current best dev=0.718) selected=False test_acc=0.869 (2.6s)


[V1 MLP] V1b balanced Attack Detection     dev_acc=0.984 (current best dev=0.988) selected=False test_acc=0.989 (4.3s)


[V1 MLP] V1b balanced Point Prediction     dev_acc=0.662 (current best dev=0.711) selected=False test_acc=0.660 (3.3s)


[V1 MLP] V1b balanced Antenna Prediction   dev_acc=0.229 (current best dev=0.249) selected=False test_acc=0.181 (4.6s)

MLP became selected_on_dev for: no task/design (existing champions still win)


,task,design,selected_on_dev,dev_pooled_acc,accuracy,ci_low,ci_high
45,Attack Detection,V1a temporal (block 6 held out),False,0.9838,0.9971,0.9892,1.0000
46,Point Prediction,V1a temporal (block 6 held out),False,0.6801,0.8687,0.7755,0.9583
47,Attack Detection,V1b balanced session hold-out,False,0.9838,0.9893,0.9705,0.9982
48,Point Prediction,V1b balanced session hold-out,False,0.6625,0.6596,0.3846,0.8958
49,Antenna Prediction,V1b balanced session hold-out,False,0.2285,0.1809,0.0217,0.4130


## 4. R2.4 — Threshold Detector as a Formal Baseline

`DecisionTreeClassifier(max_depth=1)` on `mean_rssi` alone — a single learned RSSI
threshold. Same primary protocol (leave-one-block-out); nothing to tune, so this is a
plain OOF loop (`cross_val_predict`), not `nested_cv`. Reported with the same
false-alarm-rate / detection-sensitivity breakdown as KNN in V5/F3 (Task A1).

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict

X_thr = dataset[["mean_rssi"]]
y_thr = dataset["attack"]
outer_cv_thr, groups_thr = ep.get_outer_cv_and_groups("primary", "attack", dataset)

t0 = time.time()
thr_pred = cross_val_predict(DecisionTreeClassifier(max_depth=1, random_state=ep.RANDOM_STATE), X_thr, y_thr, cv=outer_cv_thr, groups=groups_thr, n_jobs=-1)
thr_metrics = ep.classification_metrics(y_thr, thr_pred, "attack")
print(f"Threshold (mean RSSI): pooled_acc={thr_metrics['pooled_acc']:.4f} macro_f1={thr_metrics['macro_f1']:.4f} "
      f"precision={thr_metrics['precision']:.4f} recall={thr_metrics['recall']:.4f} fpr={thr_metrics['fpr']:.4f} ({time.time()-t0:.1f}s)")

thr_ci = ep.cluster_bootstrap_ci(y_thr.to_numpy(), thr_pred, dataset["session"].to_numpy(), n_boot=2000, seed=0)
print(f"95% CI (session bootstrap): [{thr_ci['ci_low']:.4f}, {thr_ci['ci_high']:.4f}]")

thr_sensitivity = ep.attack_detection_sensitivity(y_thr.to_numpy(), thr_pred, dataset["session"].to_numpy(), freq_seconds=5, window_sizes=(1, 2, 3))
thr_fa_rates, thr_fa_hours = ep.false_alarm_rate_per_hour(y_thr.to_numpy(), thr_pred, dataset["session"].to_numpy(), freq_seconds=5, window_sizes=(1, 2, 3))

threshold_detector_df = pd.DataFrame([
    {
        "window": w, "false_alarms_per_hour": thr_fa_rates[w],
        "detection_rate": thr_sensitivity[w]["detection_rate"],
        "median_delay_s": thr_sensitivity[w]["median_delay_s"],
        "row_recall": thr_sensitivity[w]["row_recall"],
        "n_attack_sessions": thr_sensitivity[w]["n_attack_sessions"],
        "n_detected": thr_sensitivity[w]["n_detected"],
    }
    for w in (1, 2, 3)
])
threshold_detector_df.attrs["pooled_acc"] = thr_metrics["pooled_acc"]
threshold_detector_df.attrs["macro_f1"] = thr_metrics["macro_f1"]
threshold_detector_df.attrs["precision"] = thr_metrics["precision"]
threshold_detector_df.attrs["recall"] = thr_metrics["recall"]
threshold_detector_df.attrs["fpr"] = thr_metrics["fpr"]
threshold_detector_df.attrs["ci_low"] = thr_ci["ci_low"]
threshold_detector_df.attrs["ci_high"] = thr_ci["ci_high"]
threshold_detector_df.attrs["attack_free_hours"] = thr_fa_hours
threshold_detector_df.to_csv(RESULTS_DIR / "threshold_detector.csv", index=False)
threshold_detector_df.round(3)

Threshold (mean RSSI): pooled_acc=0.9883 macro_f1=0.9851 precision=0.9691 recall=0.9877 fpr=0.0115 (0.2s)
95% CI (session bootstrap): [0.9837, 0.9922]


,window,false_alarms_per_hour,detection_rate,median_delay_s,row_recall,n_attack_sessions,n_detected
0,1,8.308,1.0,0.0,0.988,48,48
1,2,0.000,1.0,5.0,0.904,48,48
2,3,0.000,1.0,10.0,0.820,48,48


In [7]:
# Attack Detection row for the main table: Threshold uses mean_rssi, which
# only exists in the FE (13) feature set — Base (4) / Point / Antenna columns
# are not applicable and left as NaN.
task_label = ep.TASKS["attack"]["label"]
row = {c: np.nan for c in table1_df.columns}
row[f"{task_label} Pooled Acc (FE)"] = thr_metrics["pooled_acc"]
row[f"{task_label} Acc Mean+-Std (FE)"] = f"{thr_metrics['pooled_acc']:.3f} (OOF, no folds to average — single learned threshold)"
row[f"{task_label} Macro F1 (FE)"] = round(thr_metrics["macro_f1"], 4)

table1_df = table1_df.drop(index="Threshold (mean RSSI)", errors="ignore")
table1_df.loc["Threshold (mean RSSI)"] = pd.Series(row)
table1_df.to_csv(RESULTS_DIR / "table1_primary.csv")
print("Threshold row added to table1_primary.csv (Attack Detection, FE only; Point/Antenna/Base = N/A).")
table1_df.loc[["Threshold (mean RSSI)"], [f"{task_label} Pooled Acc (FE)", f"{task_label} Macro F1 (FE)"]]

Threshold row added to table1_primary.csv (Attack Detection, FE only; Point/Antenna/Base = N/A).


,Attack Detection Pooled Acc (FE),Attack Detection Macro F1 (FE)
Model,,
Threshold (mean RSSI),0.988268,0.9851


## 5. Acceptance Checks

- Every CI above resamples sessions, not rows (`eval_protocol.cluster_bootstrap_ci` /
  the same convention as Task A1).
- MLP's dev-only ranking for V1 reuses `select_best_model_on_dev`'s underlying
  `nested_cv`, which asserts no group overlap in every outer and inner split (F1,
  Task A1) — unchanged here.
- This notebook runs clean top-to-bottom from a fresh kernel.

In [8]:
print("table1_primary.csv models:", table1_df.index.tolist())
print("final_holdout.csv models:", sorted(final_holdout_df.model.unique()))
assert "MLP" in table1_df.index and "MLP" in final_holdout_df.model.unique()
assert "Threshold (mean RSSI)" in table1_df.index
print("R2.3/R2.4 additions verified present in both result files.")

table1_primary.csv models: ['Logistic Regression', 'SVM (RBF)', 'KNN', 'Decision Tree', 'Random Forest', 'Extra Trees', 'XGBoost', 'LightGBM', 'CatBoost', 'MLP', 'Threshold (mean RSSI)']
final_holdout.csv models: ['CatBoost', 'Decision Tree', 'Extra Trees', 'KNN', 'LightGBM', 'Logistic Regression', 'MLP', 'Random Forest', 'SVM (RBF)', 'XGBoost']
R2.3/R2.4 additions verified present in both result files.
